### LoopFlopy Example 3 - Adding Heterogeneity  
WORK IN PROGRESS DON'T USE

In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import sys
import os
import pickle
import flopy
import geopandas as gpd
from shapely.geometry import LineString,Point,Polygon,MultiPolygon,MultiPoint,shape

%load_ext autoreload
%autoreload 2

import loopflopy
import loopflopy.utils as utils

print(loopflopy.__file__)
modules = utils.list_modules(loopflopy)
for module in modules: print(module)

### 1. Setup Project

In [ ]:
# Project setup
name = 'example'
workspace = '../modelfiles/'
results = '../results/'
figures = '../figures/'
triexe = '../exe/triangle.exe'
mfexe_name = '../exe/mf6.exe'

from loopflopy.project import Project
project = Project(name, workspace, results, figures, triexe, mfexe_name)
project.crs = 28350  # UTM zone 50S, EPSG:28350

### 2. Spatial

In [ ]:
from loopflopy.spatial import Spatial
spatial = Spatial(epsg = 28350)

from preproc.spatial import make_bbox_shp, make_model_boundary
make_bbox_shp(spatial, x0 = 348000, x1 = 390000, y0 = 6503000, y1 = 6520000) # x1 = 415000
make_model_boundary(spatial, boundary_buff = 500, simplify_tolerance = 800, node_spacing = 1000)

spatial.x0, spatial.x1, spatial.y0, spatial.y1

In [ ]:
from preproc.spatial import make_obs_bores, make_geo_bores, make_pump_bores
make_obs_bores(spatial)
make_geo_bores(spatial)
make_pump_bores(spatial)

from preproc.spatial import make_head_boundary, make_faults, make_lakes
make_head_boundary(spatial)
make_faults(spatial)
make_lakes(spatial)

from preproc.spatial import make_xsections, make_pilot_points
make_xsections(spatial)
make_pilot_points(spatial)

In [ ]:
from preproc.spatial import plot_spatial_inputs
plot_spatial_inputs(spatial, 
                             obsbores = True, 
                             pilotpoints = True,
                             pumpbores = False, 
                             geobores = False, 
                             xsections = True, 
                             fault = False,
                             title = 'Observation bores, pilot points and cross-sections')

In [ ]:
plot_spatial_inputs(spatial, 
                    obsbores = False, 
                    pilotpoints = False,
                    pumpbores = False, 
                    geobores = True, 
                    controlpoints = True,
                    xsections = True, 
                    fault = False,
                    title = 'Geological bores, control points and cross-sections')

In [ ]:
# Create a map
m = spatial.obsbore_gdf.explore(location=[-31.4, 115.8], zoom_start=10, popup = 'ID')
spatial.model_boundary_gdf.explore(m=m, fill = False)
#spatial.faults_gdf.explore(m=m, color = 'red')
spatial.pumpbore_gdf.explore(m=m, color = 'red', popup = 'ID')

### 3. Structural Model

In [ ]:
# A bounding box for structural model
x0, x1 = 348000, 400000
y0, y1 = 6500000, 6550000
z0, z1 = -1000, 500
bbox = [(x0, y0, z0), (x1, y1, z1)]

from loopflopy.spatial_routines import make_bbox_shp
crop_polygon = make_bbox_shp(x0, x1, y0, y1, crs=project.crs, fname = '../data/shp/structuralmodel.shp')

geodata_fname = '../data/example_data.xlsx'
data_sheetname, strat_sheetname = 'geo_bores', 'strat'

from loopflopy.structuralmodel import StructuralModel
structuralmodel = StructuralModel(bbox, geodata_fname, data_sheetname, strat_sheetname)

from preproc.structural import prepare_strat_column, prepare_geodata, create_structuralmodel
prepare_strat_column(structuralmodel)
prepare_geodata(structuralmodel, spatial, dem_tif = "../data/dem/cropped_raster.tif",
                control = True, truth = False, Fault = False)
create_structuralmodel(structuralmodel, Fault = False)

#structuralmodel.data

In [ ]:
structuralmodel.contour_bottom(spatial, contour_interval = 10, unit = 'TQ',) 

In [ ]:
z0 = -1000 # just for plotting. Make sure within bbox of structural model
z1 = 200   # just for plotting. Make sure within bbox of structural model
nh, nz = 50, 50 # plotting resolution

for i in range(len(spatial.xsections)):
    start = spatial.xsections[i][0]
    end   = spatial.xsections[i][1]
    structuralmodel.plot_xytransect(title = spatial.xsection_names[i], start = start, end = end, z0 = z0, z1 = z1, nh = nh, nz = nz)

In [ ]:
'''def evaluate_model(structuralmodel, start, end, z0, z1, nh, nz, **kwargs):

    x0 = start[0]
    y0 = start[1]
    x1 = end[0]
    y1 = end[1]

    x = np.linspace(x0, x1, nh)
    y = np.linspace(y0, y1, nh)        
    z = np.linspace(z0, z1, nz)

    X = np.tile(x, (len(z), 1)) 
    Y = np.tile(y, (len(z), 1)) 
    Z = np.tile(z[:, np.newaxis], (1, nh))  # Repeat z along columns (nh times)

    labels = structuralmodel.strat_names[1:]
    ticks = [i for i in np.arange(0,len(labels))]
    boundaries = np.arange(-1,len(labels),1)+0.5

    a = np.array([X.flatten(),Y.flatten(),Z.flatten()]).T
    V = structuralmodel.model.evaluate_model(a).reshape(np.shape(X))

    return V

i = 0 # first transect
start = spatial.xsections[i][0]
end   = spatial.xsections[i][1]
nh, nz = 50, 50
V = evaluate_model(structuralmodel, start, end, z0, z1, nh, nz)
np.unique(V.flatten(), return_counts=True)'''

### 4. Meshing

In [ ]:
# Create mesh object
from loopflopy.mesh import Mesh
special_cells = {
                 'chd': ['east', 'west'], 
                 'wel': ['pumpbores'], 
                 'obs': ['obsbores'],
                 'poly': ['lake'], 
                 }
mesh = Mesh(plangrid = 'tri', special_cells = special_cells) # special cells to identify as arguements 

In [ ]:
from preproc.mesh import define_mesh
define_mesh(mesh)

In [ ]:
mesh.create_bore_refinement(spatial)

node_list = ['bore_refinement_nodes','lake_nodes']#fault_nodes', 'bore_refinement_nodes',]
polygon_list = ['model_boundary_poly', 'inner_boundary_poly']
mesh.prepare_nodes_and_polygons(spatial, node_list, polygon_list)
mesh.create_mesh(project, spatial)

print('number of cells in plan = ', mesh.ncpl)

In [ ]:
xlim = [spatial.x0 - 2000, spatial.x1 + 2000]
ylim = [spatial.y0 - 2000, spatial.y1 + 2000]

features = ['wel', 'obs', 'lake']
mesh.plot_cell2d(spatial, features, xlim = xlim, ylim = ylim)

### 5. Find Cell IDs of spatial features

In [ ]:
# Locate special cells and plot
mesh.locate_special_cells(spatial)

xlim = [355000, 370000]
ylim = [6525000, 6535000]

mesh.plot_feature_cells(spatial, xlim = None, ylim = None)

### 5. Topography

In [ ]:
from loopflopy.dem import DEM

# Crops large DEM to structural model bounding box
uncropped_dem = DEM(geotiff_fname = "../data/dem/Hydro_Enforced_1_Second_DEM.tif")
uncropped_dem.crop_raster(bbox_path = "../data/shp/bbox.shp", cropped_raster_path = "../data/dem/cropped_raster.tif")

### 6. Geomodel

In [ ]:
# Set up Geomodel (lithological block model of cells)
scenario = 'example'
vertgrid = 'con'    # {voxel: nlay, z0, z1} {con:   nls, res} {con2:  nls, res, max_thick}
z0, z1 = -1000, 200 

from loopflopy.geomodel import Geomodel # geomodel - pillar method + flat model bottom, geomodel2 - surface method and option to model x number of geolayers
geomodel = Geomodel(scenario, mesh, structuralmodel, vertgrid, z0, z1, nlg = 5) # surfaces calculated at a 3m resolution

In [ ]:
# Crops structuralmodel DEM to geomodel bounding box
dem = DEM(geotiff_fname = '../data/dem/cropped_raster.tif')
dem.crop_raster(bbox_path =  "../data/shp/model_boundary.shp", 
                cropped_raster_path = "../data/dem/geomodel_raster.tif")
dem.plot_geotiff()

#resample DEM to mesh
dem.resample_topo(mesh, 
                  output_fname = '../modelfiles/geomodel_topo.pkl', 
                  method="nearest", 
                  extrapolate_edges= True,) # only need to run when mesh changes. Take 1-2 minutes
dem.load_topo(fname = '../modelfiles/geomodel_topo.pkl')
dem.plot_topo(mesh, levels = np.linspace(0, 110, 10))

In [ ]:
# Create geomodel surfaces
geomodel.evaluate_structuralmodel(mesh, structuralmodel, res = 5)

In [ ]:
geomodel.create_model_layers(surface = dem.topo, nls = 2) # max_thick = 50, 

In [ ]:
for i in range(len(spatial.xsections)):
    x0, y0 = spatial.xsections[i][0]
    x1, y1 = spatial.xsections[i][1]
    print(x0, y0, x1, y1)
    geomodel.geomodel_transect_lith(figsize = (8,3), plot_node = None, y0=y0, y1 = y1, x0=x0, x1=x1)

In [ ]:
array = geomodel.botm_geo[0] # Top geo layer
title = 'TQ_thickness'
xy = [structuralmodel.data.X, structuralmodel.data.Y]
mesh.plot_array(array, title = title, levels = None, xy = xy)

In [ ]:
array = geomodel.botm_geo[1] - geomodel.botm_geo[0]# Top geo layer
title = 'Kcok thickness'
xy = [structuralmodel.data.X, structuralmodel.data.Y]
mesh.plot_array(array, title = title, levels = None, xy = xy)

### 6. Create synthetic heterogenous field

#### Method 1: Kriging between random points/values

In [ ]:
from preproc.truth import make_random_points, plot_random_points
make_random_points(spatial, num_points=100, fname = '../data/random_points.xlsx') # only need to do once
plot_random_points(spatial)

In [ ]:
from preproc.structural import evaluate_geomodel_scalarfield
scalarfield = evaluate_geomodel_scalarfield(structuralmodel, geomodel)

In [ ]:
from preproc.truth import Truth1

truth1 = Truth1()
points_df = pd.read_excel('../data/random_points.xlsx')
gdf = truth1.make_randomfield_3d(geomodel, mesh, spatial, scalarfield, points_df, units = geomodel.units)
gdf

In [ ]:
# Makes a random stochastic field for all the units with a mean of 0 and variance of 1. Take up to 5min.
anisotropy = (1., 0.5, 0.1)
corr_length = 5000
nugget = 0.05
n_neighbors = 10
rebuild_threshold = 0.1
truth1.field = truth1.stochasticfieldgeneration(gdf, geomodel, mesh, anisotropy, corr_length, nugget, n_neighbors, rebuild_threshold, scalarfield)

In [ ]:
a = truth1.field[0] - truth1.field[1]
plt.hist(a)

In [ ]:
from preproc.truth import plot_fields

plot_fields(geomodel, mesh, field=truth1.field, title="Method 1", levels=[-1, -0.5, 0, 0.5, 1], 
            vmin = -1, vmax = 1, xy=(points_df.x.values, points_df.y.values))


In [ ]:
for i in range(len(spatial.xsections)):
    x0, y0 = spatial.xsections[i][0]
    x1, y1 = spatial.xsections[i][1]
    geomodel.geomodel_transect_lith(plot_node = None, y0=y0, y1=y1, x0=x0, x1=x1, figsize = (4,2))
    geomodel.geomodel_transect_array(truth1.field, title = 'Method 1', grid = True, y0=y0, y1=y1, x0=x0, x1=x1, figsize = (4,2))   

#### Method 2: Stochastic Field from random points

In [ ]:
# Random field based on random points
# Makes a random stochastic field for all the units with a mean of 0 and variance of 1. Take up to 5min.
from preproc.truth import Truth2
truth2 = Truth2()

anisotropy = (1., 1., 0.1)
corr_length = 5000
nugget = 0.05
rebuild_threshold = 0.1
n_neighbours = 4
n_initial_points = 10
truth2.field = truth2.stochasticfieldgeneration(geomodel, mesh, anisotropy, corr_length, nugget, n_neighbours, n_initial_points, rebuild_threshold, scalarfield)

In [ ]:
plot_fields(geomodel, mesh, field=truth2.field, title="Method 2", levels=[-1, -0.5, 0, 0.5, 1], 
            vmin = -1, vmax = 1, xy=(points_df.x.values, points_df.y.values))

In [ ]:
for i in range(len(spatial.xsections)):
    x0, y0 = spatial.xsections[i][0]
    x1, y1 = spatial.xsections[i][1]
    geomodel.geomodel_transect_lith(plot_node = None, y0=y0, y1=y1, x0=x0, x1=x1, figsize = (4,2))
    geomodel.geomodel_transect_array(truth2.field, title = 'Method 2', grid = True, y0=y0, y1=y1, x0=x0, x1=x1, figsize = (4,2))   

In [ ]:
truth1.field.shape

In [ ]:
prop_df = pd.read_excel('../data/truth_properties.xlsx', sheet_name = 'truth')
prop_df


In [ ]:
truth1.make_prop_fields(geomodel, prop_df)

#prop_disv = prop_layicpl_ma.flatten() # includes mask
#prop_disu = prop_layicpl_ma.compressed() # doesnt include mask

In [ ]:
truth1.print_prop_stats()

In [ ]:
plot_fields(geomodel, mesh, field=np.log10(truth1.kh), title="Method 1 Kh", 
            vmin = -3, vmax = 2, #levels=[-1, -0.5, 0, 0.5, 1],  
            xy=(points_df.x.values, points_df.y.values))

In [ ]:
from preproc.truth import plot_propfield
unit, sublay, property = 'TQ', 0, 'kh'
plot_propfield(truth1, mesh, geomodel, spatial, unit, sublay, property, xlim = None, ylim = None)

unit, sublay, property = 'Kcok', 1, 'ss'
plot_propfield(truth1, mesh, geomodel, spatial, unit, sublay, property, xlim = None, ylim = None)

In [ ]:
#spatial.xsections = [[(355000, 6530000),(400000, 6540000)], [(365000, 6510000),(400000, 6525000)]]
x0, x1, y0, y1 = spatial.xsections[0][0][0], spatial.xsections[0][1][0], spatial.xsections[0][0][1], spatial.xsections[0][1][1]
property = 'kv'
from preproc.truth import plot_propfield_usg
plot_propfield_usg(truth1,geomodel, spatial, property, x0, x1, y0, y1)

In [ ]:
props = pd.read_excel(structuralmodel.geodata_fname, sheet_name = 'properties')
geomodel.iconvert_perlay = props.iconvert.tolist()

geomodel.fill_cell_properties_heterogeneous(truth)

### 6. Prepare Data for Flowmodel

In [ ]:
from process_data import Data
data = Data()

data.process_ic()#geomodel)
data.process_rch(geomodel, mesh)
data.process_chd(spatial, geomodel, mesh)

wel_q = [-100000, 0]
wel_qlay= [0, 0]
data.process_wel(geomodel, mesh, spatial, wel_q, wel_qlay)

#### Process Observations

In [ ]:
df = pd.read_excel('../data/example_data.xlsx', sheet_name = 'obs_bores')
import process_observations
gdf = process_observations.make_obs_gdf(df, geomodel, mesh, spatial)

from loopflopy.observations import Observations
observations = Observations(gdf)
observations.make_recarray() # makes recarray for flowmodel

print(observations.gdf)

### 8. Create and run steady-state flow model

In [ ]:
from loopflopy.flowmodel import Flowmodel
project.mfexe_name = '../exe/mf6.exe'
scenario = 'steadymodel'

fm = Flowmodel(scenario, project, data, observations, mesh, geomodel)
# Write and run flow model files
sim = fm.write_flowmodel(chd = True, wel = True, obs = True, rch = True)
fm.run_flowmodel(sim)

In [ ]:
utils.print_object_details(fm)

In [ ]:
# IF A PROBLEM NODE FROM MFSIM, CAN LOT LOCATION HERE!
#utils.plot_node(1508-1, geomodel, structuralmodel, spatial, sim, scenario, z1 = 500, features = ['fault'])

In [ ]:
watertable = fm.get_watertable(geomodel, fm.head)
fm.plot_watertable(spatial, mesh, geomodel, fm, watertable,)#, extent, vmin, vmax

In [ ]:
fm.plot_transect(spatial, structuralmodel, 'head',  y0 = 6520000, y1 = 6520000, z1 = 300)#, X0, X1, Y0, Y1)

In [ ]:
fm.plot_transect(spatial, structuralmodel, 'lith', y0 = 6520000, y1 = 6520000)

In [ ]:
obj = fm
methods = [func for func in dir(obj) if callable(getattr(obj, func)) and not func.startswith("__")]
print("Methods in the class:  ",methods)
attributes = [attr for attr in dir(obj) if not callable(getattr(obj, attr)) and not attr.startswith("__")]
print("\nAttributes in the class:  ", attributes)

#for attribute, value in flowmodel.data.__dict__.items(): print(attribute)

### Create a steady state "truth"

In [ ]:
# Get convert obs package to arrays of oberservation heads
heads_truth = fm.head

# Get times
csv_file = os.path.join('../results/ref_past.csv')
data_set = pd.read_csv(csv_file, header=0)
df = pd.DataFrame(data_set)
times_past = df.iloc[:,0]/365
csv_file = os.path.join(project.workspace, R.modelname + "_future.csv")
data_set = pd.read_csv(csv_file, header=0)
df = pd.DataFrame(data_set)
times_future = df.iloc[:,0]/365+2
times = list(times_past) + list(times_future)

# Pickle reference results
fname = P.results + 'times.pkl'
pickle.dump(times, open(os.path.join(fname),'wb'))
fname = '../results/ref_ss.pkl'
pickle.dump(R.ref_steady, open(os.path.join(fname),'wb'))
fname = P.results + 'ref_past.pkl'
pickle.dump(R.ref_past, open(os.path.join(fname),'wb'))
fname = P.results + 'ref_future.pkl'
pickle.dump(R.ref_future, open(os.path.join(fname),'wb'))


### 9. Create and run transient flow model

In [ ]:
scenario = 'transientmodel'
well_q = [-2000, -2000]
data.process_wel(geomodel, mesh, spatial, well_q, wel_qlay)
data.strt = fm.head
fm2 = Flowmodel(scenario, project, data, observations, mesh, geomodel)

perlen = 2 * 365 # length of period (days)
nts = 12         # number of time steps for this period
fm2.perioddata = [(perlen, nts, 1.1)] # perlen, nts, tsmult

In [ ]:
sim = fm2.write_flowmodel(transient = True, chd = True, wel = True, obs = True, rch = True)

In [ ]:
fm2.run_flowmodel(sim, transient = True)

In [ ]:
watertable = fm2.get_watertable(geomodel, fm2.head)
fm2.plot_watertable(spatial, mesh, geomodel, fm2, watertable,)#, extent, vmin, vmax
fm2.plot_transect(spatial, structuralmodel, 'head',  y0 = 6520000, y1 = 6520000, z1 = 300)#, X0, X1, Y0, Y1)

#### Create "truth" observations and plot

In [ ]:
# Get convert obs package to arrays of oberservation heads
R.ref_steady  = functions.process_obs_steady(P, R)
R.ref_past    = functions.process_obs_past(P, R)
R.ref_future  = functions.process_obs_future(P, R)

# Get times
csv_file = os.path.join(P.workspace, R.modelname + "_past.csv")
data_set = pd.read_csv(csv_file, header=0)
df = pd.DataFrame(data_set)
times_past = df.iloc[:,0]/365
csv_file = os.path.join(P.workspace, R.modelname + "_future.csv")
data_set = pd.read_csv(csv_file, header=0)
df = pd.DataFrame(data_set)
times_future = df.iloc[:,0]/365+2
times = list(times_past) + list(times_future)

# Pickle reference results
fname = P.results + 'ref_ss.pkl'
pickle.dump(R.ref_steady, open(os.path.join(fname),'wb'))
fname = P.results + 'ref_past.pkl'
pickle.dump(R.ref_past, open(os.path.join(fname),'wb'))
fname = P.results + 'ref_future.pkl'
pickle.dump(R.ref_future, open(os.path.join(fname),'wb'))
fname = P.results + 'times.pkl'
pickle.dump(times, open(os.path.join(fname),'wb'))

In [ ]:
# PLOT REFERENCE OBSERVATIONS
import matplotlib.pyplot as plt
colors = ['red','green','blue']
R.ref_future.shape
for j in range(P.nobs):
    for i in range(P.nzobs):
        plt.plot(times_past, R.ref_past[j][i],'o', ms = 2, color = colors[i])
        plt.plot(times_future, R.ref_future[j][i],'o', ms = 2, color = colors[i])
plt.xlabel('Time (years)')
plt.ylabel('Head (m)')

In [ ]:
# CREATE SYNTHETIC OBSERVATIONS BY PERTURBING ACTUAL HEADS
err = 0.05
hobs_ref = np.copy(R.ref_past)    
for i in range(P.nobs):
    for j in range(P.nzobs):
        for k in range(hobs_ref.shape[2]):
            hobs_ref[i][j][k] += np.random.normal(0.,err) # Perturb samples either side of measurement
plt.plot(R.ref_past.flatten(), 'o', ms = 1, label = 'actual heads')
plt.plot(hobs_ref.flatten(),'o', ms = 1, label = 'perturbed heads')
plt.xlabel('Observation index')
plt.ylabel('Head (m)')
plt.legend()

In [ ]:
%run ./figures.ipynb   
figure8(P, R)

#### Prepare PEST input arrays

In [ ]:
# PRINTING TRUTH HEADS FOR HISTORY MATCHING (PEST - HEADS.INS)
hobs_ref.shape #(5, 3, 12)
for i in range(P.nobs):
    for j in range(P.nzobs):
        for k in range(hobs_ref.shape[2]):
            print('l1 [OBS%i_%i.%i]1:20' %(i, j, k))

In [ ]:
# PRINTING TRUTH HEADS FOR HISTORY MATCHING (PEST - CONTROL.PST)
for i in range(P.nobs):
    for j in range(P.nzobs):
        for k in range(hobs_ref.shape[2]):
            print('OBS%i_%i.%i  %f  10.0  head' %(i, j, k, hobs_ref[i, j, k]))

#### Prepare parameter bounds

In [ ]:
# LOG
k_min = [0, -2, 0, -2, -1, -2.5, -0.5, -2.5, -5, -5, -5]
k_max = [1, -1, 1, -1, 0, -1.5, 0.5, -1.5, -4, -4, -4]
k_mean = [(k_max[i] - k_min[i])/2 + k_min[i] for i in range(len(k_min))]

print(k_min)
print(k_max)
print(k_mean)

In [ ]:
pickleoff = open('../results/reference.pkl','rb')
reference = pickle.load(pickleoff)
pickleoff.close()

# BLUE - MAX/MIN
# ORANGE - MEAN
# RED - ACTUAL
fig = plt.figure(figsize = (12,5))
plt.plot(np.linspace(0,10,11), k_min, 'o', color = 'blue')
plt.plot(np.linspace(0,10,11), k_max, 'o', color = 'blue')
plt.plot(np.linspace(0,10,11), k_mean, 'o', color = 'orange')
plt.plot(np.linspace(0,10,11), logfunc(reference[0] + reference[1] + reference[2][1:]), 'o', color = 'red')
plt.ylabel('Log')
plt.xticks(np.linspace(0,10,11), ['loghk0', 'loghk1', 'loghk2', 'loghk3', 
                                'logvk0', 'logvk1', 'logvk2', 'logvk3',
                                'logss1', 'logss2', 'logss2'])#, logss1, logss2, logss3, sy0, cp])
plt.show()

#### Pilot Point Setup

In [ ]:
properties = Properties() # set up a new properties object
df = pd.read_excel('../data/example_data.xlsx', sheet_name = 'pilot_points')
properties.make_gdf(geomodel, mesh, spatial, df)
#print(properties.gdf)

anisotropy = (1., 1., 0.0001)
CL = 5000.
nugget = 0.05
rebuild_threshold = 0.1

properties.kriging(geomodel, mesh, anisotropy, CL, nugget, rebuild_threshold)

#spatial.xsections = [[(355000, 6530000),(400000, 6540000)], [(365000, 6510000),(400000, 6525000)]]
x0, x1, y0, y1 = spatial.xsections[0][0][0], spatial.xsections[0][1][0], spatial.xsections[0][0][1], spatial.xsections[0][1][1]
properties.plot_propfield_usg(geomodel, spatial, x0, x1, y0, y1, property = 'kh')

'''# LOG
k_min = [0, -2, 0, -2]
k_max = [1, -1, 1, -1]
k_mean = [(k_max[i] - k_min[i])/2 + k_min[i] for i in range(len(k_min))]

print(k_min)
print(k_max)
print(k_mean)

Min.append(k_min[i+8]) 
Max.append(k_max[i+8])

# LAYER K
nr_param = 200 # Create lots of parameter sets
nr = 100 # How many successful flow models we want
import random 
logfunc = lambda e: np.log10(e)

def normal_sampling(mu, sigma, nr_param):
    nums = [] 
    count = 0
    while count < nr_param:
        temp = random.gauss(mu,sigma) 
        if temp >= mu - 3 * sigma and temp <= mu + 3*sigma:
            nums.append(temp)
            count += 1
    return(nums)

hk_mu = np.copy(hk) # from variables.ipynb
hk_mu_log = logfunc(hk_mu)
hk_sigma_log = [0.66, 0.66, 0.66, 0.66, 0.66, 0.66, 0.66, 0.66]

vk_mu = np.copy(vk) # from variables.ipynb
vk_mu_log = logfunc(vk_mu)
vk_sigma_log = [0.66, 0.66, 0.66, 0.66, 0.66, 0.66, 0.66, 0.66]

hk_parameters = []
vk_parameters = []

for n in range(8): # for each layer
    hk_parameters.append(normal_sampling(hk_mu_log[n], hk_sigma_log[n], nr_param))'''

#### History Matching

In [ ]:
def plot_scalarfield(self, transect_y, nx, nz, **kwargs):

        self.sequence_names = []
        for item in self.strat['sequence'].tolist():
            if item not in self.sequence_names:
                self.sequence_names.append(item)

        x0 = kwargs.get('x0', self.x0)
        z0 = kwargs.get('z0', self.z0)
        x1 = kwargs.get('x1', self.x1)
        z1 = kwargs.get('z1', self.z1)
        
        z = np.linspace(z0, z1, nz)
        x = np.linspace(x0, x1, nx)
        X,Z = np.meshgrid(x,z)

        labels = self.strat_names[1:]
        ticks = [i for i in np.arange(0,len(labels))]
        boundaries = np.arange(-1,len(labels),1)+0.5
        Y = np.zeros_like(X)
        Y[:,:] = transect_y[0]
        V = self.model.evaluate_model(np.array([X.flatten(),Y.flatten(),Z.flatten()]).T).reshape(np.shape(Y))
        
        for i, n in enumerate(transect_y):
            fig = plt.figure(figsize=(12, 3))
            ax = plt.subplot(len(transect_y), 1, i+1)
            

            val_above = 0
            # Evaluate scalar fields for each feature to plot contours
            for feat in self.sequence_names:
                print(feat)
                A = self.model.evaluate_feature_value(feat, np.array([X.flatten(),Y.flatten(),Z.flatten()]).T).reshape(np.shape(Y))
                print(A.shape)
                ax.imshow(np.ma.masked_where(V<0,A), origin = "lower", extent = [x0,x1,z0,z1])

            # Evaluate faults to plot
            #for fault in self.faults:
            #    F = self.model.evaluate_feature_value(fault, np.array([X.flatten(),Y.flatten(),Z.flatten()]).T).reshape(np.shape(Y))
            #    ax.contour(X, Z, F, levels = [0], colors = 'Black', linewidths=2., linestyles = 'dashed') 
            if i < (len(transect_y)-1):
                ax.set_xticks(ticks = [], labels = [])
            else:
                ax.set_xlabel('Northing (m)')

            #cbar.ax.set_yticks(ticks = ticks, labels = labels, size = 8, verticalalignment = 'center')    
            ax.set_title("y = " + str(transect_y[i]), size = 8)
            ax.set_ylabel('Elev. (mAHD)')
            ax.set_aspect('10') #0.5, 'equal'
            #plt.axis('equal')
            plt.savefig('../figures/structural_ytransects.png')
            plt.show()

plot_scalarfield(structuralmodel, transect_y, nx = 60, nz = 60, z0 = -200, )#z1 = 000) #x0 = 370000, x1 = 374000, 